In [ ]:
# ── config ───────────────────────────────────────────────────
PROCESS_PHOTO = "Kinan.Sweidan-22.JPG"  # set to None for all photos
MODEL_VISION  = "qwen2.5vl:latest"  # Call 1: image description
MODEL_CONTENT = "qwen2.5:7b"        # Call 2: fine art content & tags
MODEL_SEO     = "qwen2.5:7b"        # Call 3: SEO copy

# ── imports ───────────────────────────────────────────────────
from pathlib import Path
from PIL import Image
import base64, urllib.request, json, matplotlib.pyplot as plt

RAW_DIR = Path("./data/raw")
photos = [RAW_DIR / PROCESS_PHOTO] if PROCESS_PHOTO else sorted(
    p for ext in ("*.jpg","*.JPG") for p in RAW_DIR.glob(ext)
)

In [ ]:

# ── Call 1: vision description ────────────────────────────────
DESCRIBE_PROMPT = """These are black and white environmental portraits and street photographs taken in Chicago, Illinois.
Do not describe chromatic colors (red, blue, green, yellow, orange, purple, brown, golden) — white, black, and grey are valid tonal descriptors.

Describe this photograph for a blind person in 3-4 sentences. Be literal and specific:
- Who is in the photo, what are they doing, what are they wearing
- What is in the background — objects, architecture, street elements
- Light direction and quality, tonal contrast

IMPORTANT: Only describe what you are 100% certain is visible. If you are not sure what something is, omit it entirely — do not guess and do not include it with a qualifier like "possibly" or "appears to be". Uncertain elements must be left out. If any text (signs, business names) is clearly legible, include it — otherwise say nothing about it."""


# ── Call 2: fine art e-commerce content writer ────────────────
CONTENT_PROMPT_TEMPLATE = """You are a fine art e-commerce content writer specializing in black and white street and portrait photography. Your writing speaks to collectors and art buyers — evocative, precise, and grounded in what is literally visible.

Based on the description below, output the following fields exactly, one per line, no extra text:

CAPTION: Two sentences. First: the subject, their action, and what they wear or carry. Second: the setting, mood, and light. Write in present tense, gallery wall style — vivid but factual.
TAGS: comma-separated values ONLY from this list — person people man woman elder child street portrait architecture nature fine-art minimalist graphic dramatic mysterious solitude nostalgic joyful sad tense intimate high-contrast backlit overcast silhouette hard-light soft-light light-and-shadow neighborhood train-station bus-station park home alley skyline walking laughing waiting smoking talking running standing sitting dancing grainy gritty rusty
CONTENTLOCATION: city and neighborhood if determinable from the description, else Unknown
CONFIDENCE: integer 1-5 (how well the description supports the tags and caption)

DESCRIPTION:
{description}"""


# ── Call 3: SEO copywriter ────────────────────────────────────
SEO_PROMPT_TEMPLATE = """You are an expert SEO copywriter specializing in fine art photography prints sold online. Think about the exact terms collectors, interior designers, and photography enthusiasts type into Google or Etsy when searching for prints like this.

Based on the description below, output the following fields exactly, one per line, no extra text:

NAME: 4-8 words, compelling and search-friendly title for the print listing
SLUG: lowercase-hyphenated version of the name, suitable for a URL
KEYWORDS: exactly 12 comma-separated search terms, mix of specific (subject, location, style) and broad (e.g. "black and white photography print", "fine art wall art")

DESCRIPTION:
{description}"""


In [ ]:

import time
from IPython.display import display as ipy_display, HTML

def ollama(prompt, image_b64=None, model=MODEL_VISION):
    payload = {"model": model, "prompt": prompt, "stream": False}
    if image_b64:
        payload["images"] = [image_b64]
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())["response"]

def maybe_image(model, img_b64):
    return img_b64 if "vl" in model else None

for path in photos:
    with open(path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    # ── Call 1: vision description ────────────────────────────
    t0 = time.time()
    description = ollama(DESCRIBE_PROMPT, image_b64=img_b64, model=MODEL_VISION)
    t1 = time.time()

    # ── Call 2: fine art content ──────────────────────────────
    content = ollama(CONTENT_PROMPT_TEMPLATE.format(description=description),
                     image_b64=maybe_image(MODEL_CONTENT, img_b64), model=MODEL_CONTENT)
    t2 = time.time()

    # ── Call 3: SEO copy ──────────────────────────────────────
    seo = ollama(SEO_PROMPT_TEMPLATE.format(description=description),
                 image_b64=maybe_image(MODEL_SEO, img_b64), model=MODEL_SEO)
    t3 = time.time()

    # ── display ───────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(4, 5))
    ax.imshow(Image.open(path))
    ax.axis("off")
    ax.set_title(path.name, fontsize=9, color="#888")
    plt.tight_layout()
    plt.show()

    ipy_display(HTML(f"""
    <div style="display:grid; grid-template-columns:1fr 1fr 1fr; gap:20px; font-family:monospace; font-size:13px; max-width:1400px; margin-top:8px">
      <div>
        <div style="color:#888; font-size:11px; margin-bottom:6px">DESCRIPTION &nbsp;{MODEL_VISION} &nbsp;({t1-t0:.1f}s)</div>
        <div style="white-space:pre-wrap; line-height:1.6">{description}</div>
      </div>
      <div>
        <div style="color:#888; font-size:11px; margin-bottom:6px">CONTENT &nbsp;{MODEL_CONTENT} &nbsp;({t2-t1:.1f}s)</div>
        <div style="white-space:pre-wrap; line-height:1.8">{content}</div>
      </div>
      <div>
        <div style="color:#888; font-size:11px; margin-bottom:6px">SEO &nbsp;{MODEL_SEO} &nbsp;({t3-t2:.1f}s)</div>
        <div style="white-space:pre-wrap; line-height:1.8">{seo}</div>
      </div>
    </div>
    <div style="color:#aaa; font-size:11px; margin-top:12px">Total: {t3-t0:.1f}s &nbsp;|&nbsp; vision: {t1-t0:.1f}s &nbsp;|&nbsp; content: {t2-t1:.1f}s &nbsp;|&nbsp; seo: {t3-t2:.1f}s</div>
    <hr style="margin:20px 0; border:none; border-top:1px solid #eee">
    """))
